환경 클래스의 구조

In [1]:
import torch
import numpy as np
from FEA_Comp import fea_c
from VAE import VAE
from bc_options import *

class TopologyEnvironment:
    def __init__(self, vae_model, initial_topology):
        self.vae_model = vae_model
        self.state = initial_topology
        self.current_compliance = fea_c(64, 32, MTruss, 0, 2112, initial_topology)
    
    def reset(self):
        self.state = np.random.rand(64, 32)  # 초기 토폴로지를 무작위로 설정
        self.current_compliance = fea_c(64, 32, MTruss, 0, 2112, self.state)
        return self.state
    
    def step(self, action):
        # 현재 토폴로지를 잠재 벡터로 인코딩
        latent_vector = self.vae_model.encoder(self.state)
        
        # 잠재 벡터를 에이전트의 행동(action)으로 변형
        new_latent_vector = latent_vector + action
        
        # 새로운 잠재 벡터를 토폴로지로 디코딩
        new_topology = self.vae_model.decoder(new_latent_vector)
        
        # 새로운 토폴로지의 compliance 계산
        new_compliance = fea_c(64, 32, MTruss, 0, 2112, new_topology)
        
        # 보상 계산 (compliance가 낮을수록 보상이 큼)
        reward = self.current_compliance - new_compliance
        
        # 상태 업데이트
        self.state = new_topology
        self.current_compliance = new_compliance
        
        return self.state, reward


RL agent; MLP 모델

In [2]:
import torch.nn as nn
import torch.optim as optim
from torchrl.modules import MLP

class MLP(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_dim, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, output_dim)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# Actor-critic 모델을 위한 Value 함수와 Policy 함수 정의
class ActorCritic(nn.Module):
    def __init__(self, input_dim, action_dim):
        super(ActorCritic, self).__init__()
        self.policy = MLP(input_dim, action_dim)
        self.value = MLP(input_dim, 1)
        
    def forward(self, x):
        action_probs = self.policy(x)
        state_values = self.value(x)
        return action_probs, state_values


PPO

In [3]:
import torch
import torch.nn.functional as F

class PPO:
    def __init__(self, actor_critic, lr=3e-4, gamma=0.99, eps_clip=0.2, K_epochs=4):
        self.actor_critic = actor_critic
        self.optimizer = optim.Adam(actor_critic.parameters(), lr=lr)
        self.gamma = gamma
        self.eps_clip = eps_clip
        self.K_epochs = K_epochs
    
    def update(self, memory):
        # Memory에서 데이터를 추출
        state_arr, action_arr, reward_arr, old_log_probs_arr = memory.sample()
        
        # Advantage 계산
        state_values = self.actor_critic.value(state_arr)
        rewards = []
        discounted_reward = 0
        for reward in reversed(reward_arr):
            discounted_reward = reward + (self.gamma * discounted_reward)
            rewards.insert(0, discounted_reward)
        rewards = torch.tensor(rewards, dtype=torch.float32)
        advantages = rewards - state_values.detach()
        
        for _ in range(self.K_epochs):
            log_probs, state_values = self.actor_critic(state_arr)
            log_probs = log_probs.gather(1, action_arr)
            ratios = torch.exp(log_probs - old_log_probs_arr)
            
            surr1 = ratios * advantages
            surr2 = torch.clamp(ratios, 1 - self.eps_clip, 1 + self.eps_clip) * advantages
            
            loss = -torch.min(surr1, surr2) + F.mse_loss(state_values, rewards)
            
            self.optimizer.zero_grad()
            loss.mean().backward()
            self.optimizer.step()


학습 루프

In [4]:
from torchrl.data import ReplayBuffer

# 초기화
initial_topology = np.random.rand(64, 32)  # 초기 토폴로지

vae_model = VAE(x_dim=64*32, h_dim1= 512, h_dim2=256, z_dim=2)
vae_model.load_state_dict(torch.load('./vae_model_mtruss.pt'))

env = TopologyEnvironment(vae_model, initial_topology=initial_topology)
actor_critic = ActorCritic(input_dim=2, action_dim=2)
ppo = PPO(actor_critic)
memory = ReplayBuffer()
capacity=1000

# 학습 루프
num_episodes = 1000
for episode in range(num_episodes):
    state = env.reset()
    done = False
    
    while not done:
        action_probs, _ = actor_critic(torch.FloatTensor(state))
        action = torch.multinomial(action_probs, 1).item()
        
        new_state, reward = env.step(action)
        
        # Memory에 데이터 저장
        memory.add((state, action, reward, action_probs))
        
        state = new_state
        
        # 메모리가 가득 차면 PPO 업데이트
        if len(memory) == capacity:
            ppo.update(memory)
            memory.clear()


C:\Users\User\AppData\Local\Temp\ipykernel_12000\2747851870.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  vae_model.load_state_dict(torch.load('./vae_model_mtruss.pt')

RuntimeError: mat1 and mat2 shapes cannot be multiplied (64x32 and 2x64)